# 15 — Market power on every day: one firm, then all firms (Part 3, Stages 4–5)

**Does:** runs the strategic-bidding search from notebook 14 on all 396 days of Part 2's scored period (1 Mar 2024 – 31 Mar 2025):
- **Stage 4:** each of the 24 firms with at least 1 GW of strategic capacity bids strategically **alone**, everyone else at cost
- **Stage 5:** all 24 bid strategically **together**, until no firm wants to change its bid (best-response equilibrium)

**Input:** the Part 1 market, Part 2's weather-based availability, Stage 1's RSI and tightness. **Runs:** `src/run_part3.py` (Stage 4 ~16 min, Stage 5 ~60 min on 9 processes); this notebook reads the results.

**Plan:** `docs/part3_plan.md`, Stages 4–5. **Logged in:** `docs/model_log.md`, Step 25.

## Settings (Step 25)

| Setting | Value | Why |
|---|---|---|
| Strategy | **Bid only** (cost + adder), no withholding | Stage 3: raising the bid always beat withholding and the joint search never beat the bid alone; 4× faster |
| Players | 24 firms with ≥ 1 GW strategic capacity; smaller firms bid cost | Plan's computing budget |
| Equilibrium | Start at cost; firms in turn (largest first) play their best response; a firm moves only if it gains > Rs 1,000/h; stop when nobody moves, a set of bids repeats (cycle), or after 5 rounds | Plan, The game |
| Days | All 396 | Timing allowed it, so no sample was needed |

**Hypotheses, written before running:**
1. Alone, UP marks up on most days, not only pivotal ones (Stage 3 found +21% even on the slack day)
2. Together, firms compete: equilibrium markups are **smaller** than the largest single-firm markup
3. Markups are largest on tight days and near coal-tier steps

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dispatch import PROCESSED
import run_part3

def load(file, stage):
    """Read a Part 3 result; run its stage first if the file is missing (see src/run_part3.py)."""
    path = PROCESSED / file
    if not path.exists():
        print(f"{file} missing: running {stage} (see src/run_part3.py for run times)")
        run_part3.STAGES[stage]()
    return pd.read_csv(path, parse_dates=["Date"])

structure = pd.read_csv(PROCESSED / "market_structure_daily.csv", parse_dates=["Date"])
national = pd.read_csv(PROCESSED / "market_structure_national_daily.csv", parse_dates=["Date"]).set_index("Date")
crore = 1e7

single = load("strategic_single_daily.csv", "stage4")
eq = load("strategic_eq_national_daily.csv", "stage5")
eq_firms = load("strategic_eq_firms_daily.csv", "stage5")
eq_states = load("strategic_eq_state_daily.csv", "stage5")

single["Consumer_change_pct"] = 100 * (single["Consumer_cost_best"] / single["Consumer_cost_cost"] - 1)
eq["Consumer_change_pct"] = 100 * (eq["Consumer_cost_eq"] / eq["Consumer_cost_cost"] - 1)
print(f"Stage 4: {single['Date'].nunique()} days x {single['Firm'].nunique()} firms;  Stage 5: {len(eq)} days")

## Stage 4 — each firm alone

In [ ]:
s = single.assign(Marks_up=single["Adder"] > 0,
                  Extra_consumer=(single["Consumer_cost_best"] - single["Consumer_cost_cost"]) / crore,
                  Extra_profit=(single["Profit_best"] - single["Profit_cost"]) / crore)
by_firm = s.groupby("Firm").agg(Days_marking_up=("Marks_up", "sum"),
                                Avg_adder=("Adder", "mean"),
                                Consumers_pct=("Consumer_change_pct", "mean"),
                                Consumers_max_pct=("Consumer_change_pct", "max"),
                                Extra_consumer_cr_h=("Extra_consumer", "mean"),
                                Extra_profit_cr_h=("Extra_profit", "mean"))
by_firm["Consumer_loss_per_Rs_gain"] = by_firm["Extra_consumer_cr_h"] / by_firm["Extra_profit_cr_h"]
by_firm.sort_values("Consumers_pct", ascending=False).round(2)

### 4.2 UP alone: when does it pay to mark up?

In [ ]:
up = s[s["Firm"] == "NR UP"].set_index("Date")
up = up.join(national[["Tightness", "Near_step"]]).join(
    structure[structure["Firm"] == "NR UP"].set_index("Date")[["RSI"]])
up["Spare_band"] = pd.cut(up["Tightness"], [0, 0.10, 0.20, 0.30, 0.45, 1.0],
                          labels=["<10%", "10-20%", "20-30%", "30-45%", ">45%"])

print("UP alone, consumer cost change by spare capacity and near-step flag (mean %, days):")
display(up.pivot_table(index="Spare_band", columns="Near_step", values="Consumer_change_pct",
                       aggfunc=["mean", "count"], observed=True).round(1))
print("Correlation of UP's consumer impact with spare capacity:",
      round(up["Consumer_change_pct"].corr(up["Tightness"]), 2), " with UP's RSI:",
      round(up["Consumer_change_pct"].corr(up["RSI"]), 2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sc = ax[0].scatter(100 * up["Tightness"], up["Consumer_change_pct"], c=up["Near_step"], cmap="coolwarm", s=12)
ax[0].set_xlabel("Spare capacity (%)"); ax[0].set_ylabel("Consumer cost change (%)")
ax[0].set_title("UP alone: impact vs spare capacity (red = near a coal step)")
m = up["Consumer_change_pct"].resample("MS").mean()
ax[1].bar(m.index.strftime("%b %y"), m.values, color="tab:red")
ax[1].set_title("UP alone: average consumer cost change by month (%)")
ax[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

## Stage 5 — all firms together (equilibrium)

In [ ]:
print("Outcome of the best-response loop:"); display(eq["Status"].value_counts())
print("Rounds:", eq["Rounds"].describe()[["mean", "max"]].round(1).to_dict())

eq["Markup_pct"] = 100 * (eq["Price_avg_eq"] / eq["Price_avg_cost"] - 1)
eq["Extra_consumer_cr_h"] = (eq["Consumer_cost_eq"] - eq["Consumer_cost_cost"]) / crore
eq["Extra_profit_cr_h"] = (eq["Profit_total_eq"] - eq["Profit_total_cost"]) / crore
eq["Gen_cost_change_pct"] = 100 * (eq["Generation_cost_eq"] / eq["Generation_cost_cost"] - 1)

up_alone = up["Consumer_change_pct"]
max_alone = s.groupby("Date")["Consumer_change_pct"].max()
cmp = pd.DataFrame({"UP alone": up_alone.describe(), "Largest single firm": max_alone.describe(),
                    "All firms (equilibrium)": eq.set_index("Date")["Consumer_change_pct"].describe()}).T
print("Consumer cost change (%) over 396 days:")
display(cmp[["mean", "50%", "max"]].round(1))

summary = {
    "Days with a markup > 5%": int((eq["Markup_pct"] > 5).sum()),
    "Average markup (%)": eq["Markup_pct"].mean(),
    "Average firms marking up": eq["Firms_marking_up"].mean(),
    "Extra paid by consumers (Rs crore per peak hour, average)": eq["Extra_consumer_cr_h"].mean(),
    "Extra profit to all generators (Rs crore/h, average)": eq["Extra_profit_cr_h"].mean(),
    "True generation cost change (%, average)": eq["Gen_cost_change_pct"].mean(),
    "Unserved MW (max)": eq["Unserved_MW_eq"].max(),
}
pd.Series(summary).round(2)

In [ ]:
e = eq.set_index("Date").join(national[["Tightness", "Near_step", "Lowest_RSI", "Pivotal_firms"]])
e["Spare_band"] = pd.cut(e["Tightness"], [0, 0.10, 0.20, 0.30, 0.45, 1.0],
                         labels=["<10%", "10-20%", "20-30%", "30-45%", ">45%"])
print("Equilibrium markup (%) by spare capacity and near-step flag (mean, days):")
display(e.pivot_table(index="Spare_band", columns="Near_step", values="Markup_pct",
                      aggfunc=["mean", "count"], observed=True).round(1))
print("Markup on days with a pivotal firm vs without:")
display(e.groupby(e["Pivotal_firms"] > 0)["Markup_pct"].agg(["mean", "median", "count"]).round(1))

fig, ax = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
ax[0].plot(e.index, e["Price_avg_cost"], label="Everyone bids cost", color="tab:gray")
ax[0].plot(e.index, e["Price_avg_eq"], label="Equilibrium (24 strategic firms)", color="tab:red")
ax[0].set_ylabel("Average price (Rs/MWh)"); ax[0].legend()
ax[1].bar(e.index, e["Extra_consumer_cr_h"], color="tab:red", width=1)
ax[1].set_ylabel("Extra consumer cost\n(Rs crore per peak hour)")
plt.tight_layout(); plt.show()

In [ ]:
f = eq_firms.assign(Marks_up=eq_firms["Adder"] > 0,
                    Gain_cr_h=(eq_firms["Profit_eq"] - eq_firms["Profit_cost"]) / crore)
print("Who marks up in equilibrium:")
display(f.groupby("Firm").agg(Days_marking_up=("Marks_up", "sum"), Avg_adder_when_up=("Adder", lambda a: a[a > 0].mean()),
                              Avg_gain_cr_h=("Gain_cr_h", "mean"))
         .sort_values("Days_marking_up", ascending=False).head(12).round(2))

st = eq_states.assign(Change=eq_states["Price_eq"] - eq_states["Price_cost"])
print("Where prices move (average change Rs/MWh, largest 8 and smallest 4):")
mv = st.groupby("State")["Change"].mean().sort_values(ascending=False)
display(pd.concat([mv.head(8), mv.tail(4)]).round(0))
print("Spread of prices across states, average (max - min, Rs/MWh): at cost",
      round(st.groupby("Date")["Price_cost"].agg(np.ptp).mean()), " equilibrium",
      round(st.groupby("Date")["Price_eq"].agg(np.ptp).mean()))

## Stages 4–5 conclusions

_(filled in after the run — see below)_